In [2]:
import os
import copy
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
import random
from typing import List, Dict, Tuple, Optional
import json
import requests
from datasets import Dataset
from ragas.evaluation import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_recall,
    answer_correctness,
)
try:
    # newer RAGAS
    from ragas.metrics import semantic_similarity  # preferred
    _SIM_METRIC_IS = "semantic_similarity"
except ImportError:
    # older RAGAS
    from ragas.metrics import answer_similarity as semantic_similarity
    _SIM_METRIC_IS = "answer_similarity"

In [3]:
def set_seed(seed: int):
    """Set seeds everywhere for reproducibility (as much as PyTorch allows)."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True)
        os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":16:8")
    except Exception:
        pass

In [4]:
class OllamaClient:
    def __init__(self, base_url: str = "http://localhost:11434", model: str = "llama3.2:1b"):
        self.base_url = base_url
        self.model = model
        self._test_connection()
        
    def _test_connection(self):
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            if response.status_code == 200:
                models = response.json().get('models', [])
                model_names = [m.get('name', '') for m in models]
                if not any(self.model in name for name in model_names):
                    print(f"⚠️  Model {self.model} not found. Available models: {model_names}")
                    print(f"🔧 Run: ollama pull {self.model}")
                else:
                    print(f"✅ Ollama connection successful. Model {self.model} is available.")
            else:
                print(f"⚠️  Ollama server responded with status: {response.status_code}")
        except Exception as e:
            print(f"⚠️  Could not connect to Ollama: {e}")
            print("🔧 Make sure Ollama is running: ollama serve")
        
    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 1000, seed: Optional[int] = None) -> str:
        """Generate response using Ollama API. If seed is provided, make generation deterministic."""
        try:
            url = f"{self.base_url}/api/generate"
            options = {
                "temperature": temperature,
                "num_predict": max_tokens
            }
            if seed is not None:
                options["seed"] = int(seed)
            payload = {
                "model": self.model,
                "prompt": prompt,
                "stream": False,
                "options": options,
            }
            
            response = requests.post(url, json=payload, timeout=120)
            response.raise_for_status()
            result = response.json()
            return result.get("response", "").strip()
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 404:
                print(f"❌ Model {self.model} not found. Please run: ollama pull {self.model}")
                return "Error: Model not found. Please install the model first."
            else:
                print(f"❌ HTTP Error: {e}")
                return f"Error: HTTP {e.response.status_code}"
        except requests.exceptions.RequestException as e:
            print(f"❌ Connection error: {e}")
            print("🔧 Make sure Ollama is running: ollama serve")
            return "Error: Could not connect to Ollama. Please ensure Ollama service is running."
        except Exception as e:
            print(f"❌ Unexpected error: {e}")
            return "Error: Unexpected error occurred."


In [5]:
class EnhancedGNNRecommendationSystem:
    def __init__(self, csv_path: str, llm_client: Optional[OllamaClient] = None):
        self.df = pd.read_csv(csv_path)
        self.model_embed = SentenceTransformer('all-MiniLM-L6-v2')
        self.llm_client = llm_client or OllamaClient()
        self.setup_data()
        self.setup_models()
        
    def setup_data(self):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Using device: {self.device}")
        
        def map_performance(perf):
            if "Above 80%" in str(perf):
                return 4
            elif "Between 65-80%" in str(perf):
                return 3
            elif "Between 50-65%" in str(perf):
                return 2
            elif "Less than 50%" in str(perf):
                return 1
            return 0
        
        def map_stress(stress):
            stress_str = str(stress).lower()
            if "frequently" in stress_str or "daily" in stress_str:
                return 3
            elif "occasionally" in stress_str or "weekly" in stress_str:
                return 2
            elif "rarely" in stress_str:
                return 1
            return 0
            
        def map_confidence(conf):
            conf_str = str(conf).lower()
            if "very confident" in conf_str:
                return 3
            elif "confident" in conf_str:
                return 2
            elif "slightly confident" in conf_str:
                return 1
            return 0
        
        self.df["performance_score"] = self.df["What was your performance in previous academic levels?"].map(map_performance)
        self.df["stress_score"] = self.df["How often do you feel stressed about academics?"].map(map_stress)
        self.df["confidence_score"] = self.df["How confident are you in your academic abilities?"].map(map_confidence)
        self.combined_answers = self.df.apply(lambda row: ' '.join(row.values.astype(str)), axis=1)
        
        with torch.no_grad():
            embeddings = self.model_embed.encode(self.combined_answers.tolist(), convert_to_tensor=True)
        embeddings = embeddings.cpu()
        
        similarity_matrix = cosine_similarity(embeddings.numpy())
        threshold = 0.6
        edge_index = []
        for i in range(len(similarity_matrix)):
            for j in range(len(similarity_matrix)):
                if similarity_matrix[i][j] > threshold and i != j:
                    edge_index.append([i, j])
        
        perf = torch.tensor(self.df["performance_score"].values, dtype=torch.float).unsqueeze(1)
        stress = torch.tensor(self.df["stress_score"].values, dtype=torch.float).unsqueeze(1)
        conf = torch.tensor(self.df["confidence_score"].values, dtype=torch.float).unsqueeze(1)
        x = torch.cat([embeddings, perf, stress, conf], dim=1)
        
        self.edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous().to(self.device)
        self.features = x.to(self.device)
        self.data = Data(x=self.features, edge_index=self.edge_index)
        
    def setup_models(self):
        class GNN(torch.nn.Module):
            def __init__(self, in_channels):
                super(GNN, self).__init__()
                self.conv1 = GATConv(in_channels, 64, heads=4)
                self.conv2 = GATConv(64*4, in_channels, heads=1)

            def forward(self, x, edge_index):
                x = F.relu(self.conv1(x, edge_index))
                return self.conv2(x, edge_index)
        
        self.gnn_model = GNN(in_channels=self.features.shape[1]).to(self.device)
    
    def train_gnn(self, epochs=100):
        optimizer = torch.optim.Adam(self.gnn_model.parameters(), lr=0.01)
        loss_fn = torch.nn.MSELoss()
        self.gnn_model.train()
        for epoch in range(epochs):
            optimizer.zero_grad()
            out = self.gnn_model(self.data.x, self.data.edge_index)
            loss = loss_fn(out, self.features)
            loss.backward()
            optimizer.step()
            if epoch % 20 == 0:
                print(f"Epoch {epoch}, Loss: {loss.item():.7f}")

    def get_recommendations(self, query: str, k: int = 5) -> Tuple[List[int], List[float], List[str]]:
        query_embed = self.model_embed.encode([query], convert_to_tensor=True).cpu()
        avg_perf = self.df["performance_score"].mean()
        avg_stress = self.df["stress_score"].mean() 
        avg_conf = self.df["confidence_score"].mean()
        query_features = torch.cat([
            query_embed, 
            torch.tensor([[avg_perf, avg_stress, avg_conf]])
        ], dim=1).to(self.device)
        self.gnn_model.eval()
        with torch.no_grad():
            learned = self.gnn_model(self.data.x, self.data.edge_index)
        q = query_features.cpu()
        L = learned.cpu()
        emb_sim = cosine_similarity(q[:, :-3].numpy(), L[:, :-3].numpy())[0]
        feat_sim = cosine_similarity(q[:, -3:].numpy(), L[:, -3:].numpy())[0]
        combined = 0.8 * emb_sim + 0.2 * feat_sim
        perf_boost = self.df["performance_score"].values / 4.0 * 0.1
        conf_boost = self.df["confidence_score"].values / 3.0 * 0.1
        final_scores = combined + perf_boost + conf_boost
        top_idx = self.get_diverse_recommendations(final_scores, k)
        top_scores = final_scores[top_idx]
        rec_answers = [self.combined_answers.iloc[idx] for idx in top_idx]
        return top_idx.tolist(), top_scores.tolist(), rec_answers
    
    def get_diverse_recommendations(self, scores: np.ndarray, k: int) -> np.ndarray:
        selected, order = [], np.argsort(scores)[::-1]
        for idx in order:
            if len(selected) >= k:
                break
            if not selected:
                selected.append(idx)
                continue
            cur_p = self.df.iloc[idx]["performance_score"]
            cur_s = self.df.iloc[idx]["stress_score"]
            is_diverse = True
            for j in selected:
                if abs(cur_p - self.df.iloc[j]["performance_score"]) <= 1 and \
                   abs(cur_s - self.df.iloc[j]["stress_score"]) <= 1:
                    is_diverse = False
                    break
            if is_diverse:
                selected.append(idx)
        while len(selected) < k and len(selected) < len(order):
            for idx in order:
                if idx not in selected:
                    selected.append(idx)
                    break
        return np.array(selected)
    def prepare_context_for_llm(self, query: str, recommended_answers: List[str]) -> str:
        parts = []
        for i, ans in enumerate(recommended_answers, 1):
            clean = ans.replace('nan', '').strip()
            if len(clean) > 500:
                clean = clean[:500] + "..."
            parts.append(f"Student Response {i}: {clean}")    
        context = "\n\n".join(parts)
        
        prompt = f"""You are an expert educational advisor analyzing successful student strategies. 

Student Question: "{query}"

Based on these proven strategies from similar students:
{context}

Provide a comprehensive, actionable response following this structure:

IMMEDIATE ACTIONS (next 24-48 hours):
- List 2-3 specific steps the student can take immediately

WEEKLY STRATEGIES (ongoing implementation):
- Provide 3-4 evidence-based techniques with clear implementation steps
- Include specific time allocations where relevant

LONG-TERM APPROACH (building sustainable habits):
- Suggest 2-3 systematic changes for lasting improvement
- Explain why these strategies work based on learning science

Keep each point concise but specific. Focus on practical implementation rather than general advice.

Response:"""
        
        return prompt
    
    def get_llm_summary(self, query: str, recommended_answers: List[str], seed: Optional[int] = None) -> str:
        prompt = self.prepare_context_for_llm(query, recommended_answers)
        return self.llm_client.generate(
            prompt,
            temperature=0.0 if seed is not None else 0.3,
            max_tokens=800,
            seed=seed,
        )

In [6]:
class RAGASEvaluationFramework:
    def __init__(self, recommendation_system: EnhancedGNNRecommendationSystem, ground_truth_path: str, seed: Optional[int] = None):
        self.rec_sys = recommendation_system
        self.seed = seed
        self.load_ground_truth(ground_truth_path)
        
    def load_ground_truth(self, json_path: str):
        with open(json_path, 'r') as f:
            all_data = json.load(f)
        self.ground_truth_data = all_data[:20]
        print(f"Loaded {len(all_data)} total examples, using first {len(self.ground_truth_data)} for evaluation")
    
    def convert_user_input_to_query(self, user_input: Dict) -> str:
        main_challenge = user_input.get("Biggest challenge", "")
        academic_challenges = user_input.get("Main academic challenges", "")
        project_challenge = user_input.get("Long-term project challenge", "")
        online_challenge = user_input.get("Online learning challenge", "")
        challenges = [c for c in [main_challenge, academic_challenges, project_challenge, online_challenge] 
                      if c and c.strip() and c != "nan"]
        stress_level = user_input.get('Stress frequency', 'occasionally')
        motivation = user_input.get('Motivation', 'moderate')
        if challenges:
            primary = challenges[:2]
            query = (
                f"My main academic challenges are: {', '.join(primary)}. "
                f"I feel stressed {stress_level} and my motivation is {motivation}. "
                "What specific, actionable strategies can help me improve my academic performance and overcome these challenges?"
            )
        else:
            query = (
                f"I experience academic stress {stress_level} with {motivation} motivation. "
                "What evidence-based study strategies and stress management techniques would be most effective for me?"
            )
        return query
    
    # --------- GNN-only metrics (no LLM) ---------
    def evaluate_gnn_recommendations_with_ground_truth(self) -> Dict:
        print("🎯 Evaluating GNN Recommendations with Ground Truth (no LLM)...")
        precision_scores, recall_scores, ndcg_scores = [], [], []
        for item in self.ground_truth_data:
            user_input = item["user_input"]
            ground_truth = item["ground_truth"]
            query = self.convert_user_input_to_query(user_input)
            _, _, recommended_answers = self.rec_sys.get_recommendations(query, k=5)
            gt_words = set(ground_truth.lower().split())
            relevant_keywords = [w for w in gt_words if len(w) > 3][:10]
            if not relevant_keywords:
                continue
            relevance_scores = []
            for ans in recommended_answers:
                aw = set(ans.lower().split())
                rel = len(set(relevant_keywords) & aw) / len(relevant_keywords)
                relevance_scores.append(min(rel, 1.0))
            binary = [1 if s >= 0.2 else 0 for s in relevance_scores]
            if binary:
                precision_scores.append(sum(binary) / len(binary))
                recall_scores.append(sum(binary) / min(len(relevant_keywords), 5))
                if relevance_scores:
                    ndcg = ndcg_score([relevance_scores], [relevance_scores], k=min(len(relevance_scores), 5))
                    ndcg_scores.append(ndcg)
        return {
            "precision_at_k": float(np.mean(precision_scores)) if precision_scores else 0.0,
            "recall_at_k": float(np.mean(recall_scores)) if recall_scores else 0.0,
            "ndcg_at_k": float(np.mean(ndcg_scores)) if ndcg_scores else 0.0,
            "total_evaluated": len(precision_scores)
        }

    # --------- RAGAS (LLM) ---------
    def evaluate_with_ragas(self) -> Dict:
        print("🔍 Running RAGAS Evaluation with Ground Truth (LLM on best GNN)...")
        questions, answers, contexts, ground_truths = [], [], [], []
        for item in self.ground_truth_data:
            user_input = item["user_input"]
            ground_truth = item["ground_truth"]
            query = self.convert_user_input_to_query(user_input)
            questions.append(query)
            ground_truths.append(ground_truth)
            try:
                _, _, recommended_answers = self.rec_sys.get_recommendations(query, k=3)
                contexts.append(recommended_answers)
                llm_answer = self.rec_sys.get_llm_summary(query, recommended_answers, seed=self.seed)
                answers.append(llm_answer)
                print(f"✅ Processed query: {query[:100]}...")
            except Exception as e:
                print(f"❌ Error processing query: {e}")
                contexts.append(["No context available"])
                answers.append("Unable to generate answer")
        data = {"question": questions, "answer": answers, "contexts": contexts, "ground_truth": ground_truths}
        dataset = Dataset.from_dict(data)
        try:
            print("🚀 Running RAGAS metrics evaluation...")
            result = evaluate(dataset, metrics=[faithfulness, answer_relevancy, semantic_similarity, context_recall, answer_correctness])
            sim_key = "semantic_similarity" if "semantic_similarity" in result else ("answer_similarity" if "answer_similarity" in result else None)
            return {
                "faithfulness": result["faithfulness"],
                "answer_relevancy": result["answer_relevancy"],
                "semantic_similarity": (result[sim_key] if sim_key else float("nan")),
                "context_recall": result["context_recall"],
                "answer_correctness": result["answer_correctness"],
            }
        except Exception as e:
            print(f"⚠️  RAGAS evaluation failed: {e}")
            return self.manual_evaluation_fallback(questions, answers, contexts, ground_truths)
    
    def manual_evaluation_fallback(self, questions: List[str], answers: List[str], contexts: List[List[str]], ground_truths: List[str]) -> Dict:
        print("🔧 Running manual evaluation (fallback)...")
        def semantic_sim(a: str, b: str) -> float:
            try:
                from sklearn.feature_extraction.text import TfidfVectorizer
                from sklearn.metrics.pairwise import cosine_similarity as _cos
                vec = TfidfVectorizer(min_df=1, ngram_range=(1,2)).fit([a, b])
                A = vec.transform([a]); B = vec.transform([b])
                sim = float(_cos(A, B)[0,0])
                return max(0.0, min(1.0, sim))
            except Exception:
                sa, sb = set(a.lower().split()), set(b.lower().split())
                if not sa or not sb:
                    return 0.0
                return len(sa & sb) / len(sa | sb)
        scores = {"faithfulness": [], "answer_relevancy": [], "semantic_similarity": [], "context_recall": [], "answer_correctness": []}
        for q, a, ctx, gt in zip(questions, answers, contexts, ground_truths):
            qw, aw, gt_w = set(q.lower().split()), set(a.lower().split()), set(gt.lower().split())
            relevancy = len(qw & aw) / len(qw) if qw else 0
            scores["answer_relevancy"].append(min(relevancy, 1.0))
            correctness = len(gt_w & aw) / len(gt_w) if gt_w else 0
            scores["answer_correctness"].append(min(correctness * 2, 1.0))
            ctx_words = set(" ".join(ctx).lower().split())
            ctx_gt = len(gt_w & ctx_words) / len(gt_w) if gt_w else 0
            scores["context_recall"].append(min(ctx_gt, 1.0))
            ans_ctx = len(aw & ctx_words) / len(aw) if aw else 0
            scores["faithfulness"].append(min(ans_ctx, 1.0))
            scores["semantic_similarity"].append(semantic_sim(a, gt))
        return {m: float(np.mean(v)) if v else 0.0 for m, v in scores.items()}

In [7]:
def gnn_score_from_results(gnn_results: Dict) -> float:
    """Combine GNN metrics into a single score for model selection.
    Simple average of precision@k, recall@k, ndcg@k (all in [0,1])."""
    vals = []
    for key in ("precision_at_k", "recall_at_k", "ndcg_at_k"):
        v = gnn_results.get(key)
        if isinstance(v, (int, float)):
            vals.append(float(v))
    return float(np.mean(vals)) if vals else 0.0

In [8]:
print("🦙 Initializing Ollama client...")
ollama_client = OllamaClient(model="llama3.2:latest")

🦙 Initializing Ollama client...
✅ Ollama connection successful. Model llama3.2:latest is available.


In [9]:
print("🧠 Setting up GNN Recommendation System...")
rec_system = EnhancedGNNRecommendationSystem('Dataset3.csv', ollama_client)

🧠 Setting up GNN Recommendation System...
Using device: cuda


In [10]:
seeds_to_try = list(range(1, 51))  # 50 seeds: 1..50
epochs = 100

evaluator = RAGASEvaluationFramework(rec_system, 'rag_eval_pairs.json', seed=None)

best = {"seed": None, "score": -1.0, "state_dict": None, "gnn_results": None}

for s in seeds_to_try:
    print("" + "-"*64)
    print(f"🌱 Training & GNN-only evaluating with seed={s}")
    print("-"*64)
    set_seed(s)
    rec_system.setup_models()  # fresh init per seed
    print("🎓 Training GNN...")
    rec_system.train_gnn(epochs=epochs)
    gnn_results = evaluator.evaluate_gnn_recommendations_with_ground_truth()
    score = gnn_score_from_results(gnn_results)
    print(f"📈 Seed {s} GNN-score: {score:.4f} | details: {gnn_results}")
    if score > best["score"]:
        print(f"⭐ New best GNN found at seed {s} (score {score:.4f})")
        best.update({
            "seed": s,
            "score": float(score),
            "state_dict": copy.deepcopy(rec_system.gnn_model.state_dict()),
            "gnn_results": gnn_results,
        })

if best["state_dict"] is None:
    raise RuntimeError("No best model selected during GNN seed sweep.")

# Load best GNN back
rec_system.gnn_model.load_state_dict(best["state_dict"])
torch.save(rec_system.gnn_model.state_dict(), "best_gnn_model.pt")
with open("best_gnn_sweep_results.json", "w") as f:
    json.dump({"best_seed": best["seed"], "best_gnn_score": best["score"], "gnn_results": best["gnn_results"]}, f, indent=2)

print("🎉 Best GNN saved to best_gnn_model.pt")
print(f"🏅 Best seed: {best['seed']} | GNN score: {best['score']:.4f}")


Loaded 500 total examples, using first 20 for evaluation
----------------------------------------------------------------
🌱 Training & GNN-only evaluating with seed=1
----------------------------------------------------------------
🎓 Training GNN...
Epoch 0, Loss: 0.0573785
Epoch 20, Loss: 0.0078032
Epoch 40, Loss: 0.0072401
Epoch 60, Loss: 0.0071858
Epoch 80, Loss: 0.0071779
🎯 Evaluating GNN Recommendations with Ground Truth (no LLM)...
📈 Seed 1 GNN-score: 0.7933 | details: {'precision_at_k': 0.6900000000000001, 'recall_at_k': 0.6900000000000001, 'ndcg_at_k': 1.0, 'total_evaluated': 20}
⭐ New best GNN found at seed 1 (score 0.7933)
----------------------------------------------------------------
🌱 Training & GNN-only evaluating with seed=2
----------------------------------------------------------------
🎓 Training GNN...
Epoch 0, Loss: 0.0534425
Epoch 20, Loss: 0.0077495
Epoch 40, Loss: 0.0072294
Epoch 60, Loss: 0.0071855
Epoch 80, Loss: 0.0071782
🎯 Evaluating GNN Recommendations with

In [13]:
import json

with open("best_gnn_sweep_results.json") as f:
    best_gnn = json.load(f)
print("Best seed:", best_gnn["best_seed"])
print("GNN-only metrics:", best_gnn["gnn_results"])

with open("best_gnn_ragas_results.json") as f:
    ragas = json.load(f)
print("RAGAS metrics:", ragas["ragas_results"])

Best seed: 23
GNN-only metrics: {'precision_at_k': 0.9099999999999999, 'recall_at_k': 0.9099999999999999, 'ndcg_at_k': 1.0, 'total_evaluated': 20}
RAGAS metrics: {'faithfulness': 0.3938821756153025, 'answer_relevancy': 0.397845630676513, 'semantic_similarity': 0.12381109327676283, 'context_recall': 0.5224822670954883, 'answer_correctness': 0.5564666528809408}


In [ ]:
llm_seed = 4242
evaluator.seed = llm_seed
print("🧪 Running RAGAS (LLM) evaluation on the BEST GNN model only...")
ragas_results = evaluator.evaluate_with_ragas()

with open("best_gnn_ragas_results.json", "w") as f:
    json.dump({"llm_seed": llm_seed, "ragas_results": ragas_results}, f, indent=2)

print("📄 RAGAS Summary Quality Metrics (best GNN):")
for metric, score in ragas_results.items():
    if isinstance(score, (int, float)):
        print(f"  {metric}: {score:.4f}")
    else:
        print(f"  {metric}: {score}")

print("Done: Phase 1 picked the best GNN via 50-seed sweep; Phase 2 evaluated LLM once on that model.")

🧪 Running RAGAS (LLM) evaluation on the BEST GNN model only...
🔍 Running RAGAS Evaluation with Ground Truth (LLM on best GNN)...
✅ Processed query: My main academic challenges are: Procrastination on assignments, Frequent group projects. I feel str...
✅ Processed query: My main academic challenges are: Keeping focus in long lectures, Passive lecture formats. I feel str...
✅ Processed query: My main academic challenges are: Keeping focus in long lectures, Large reading load. I feel stressed...
✅ Processed query: My main academic challenges are: Understanding dense textbook readings, Passive lecture formats. I f...
✅ Processed query: My main academic challenges are: Time management across multiple courses, Large reading load. I feel...
✅ Processed query: My main academic challenges are: Balancing labs with part-time job, Online assessments. I feel stres...
✅ Processed query: My main academic challenges are: Balancing labs with part-time job, Passive lecture formats. I feel ...
✅ Processe